# Infra-Bench v1 — Spatial-Block-Aware Split Verification

Builds the spatial split artifact that all four FM notebooks (CROMA v2,
SatlasS2 v2, SatlasS1 v2, AlphaEarth v2) will load. The split is fixed
once verified; training-seed variation (314, 271, 161) only varies
linear-probe head init + DataLoader shuffle.

## What this notebook does

1. Builds the canonical asset list by walking the 28 `_v1_1k` manifests
   on Drive.
2. Drops any tile missing from the AlphaEarth embeddings parquet — so
   the four FMs evaluate on exactly the same tile set.
3. Projects centroids to Equal Earth (EPSG:8857), assigns each tile to a
   200 km block.
4. Within each region, greedily assigns blocks to train / val / test
   targeting 70 / 15 / 15 of tiles (seed=42).
5. Runs five verifications:
   - per-region distribution table
   - per-class distribution table (**catalogs ALL missing-from-split**
     classes — does not stop on the first)
   - comparison against the old random stratified split (regenerated
     in-notebook for an exact match against the v1 CROMA logic)
   - world-scatter plot coloured by split
   - AlphaEarth-missing tile distribution
6. Saves `asset_id_to_split_v1.parquet` to Drive **only if all
   verifications pass.**

If verification fails, the notebook prints the full failure pattern and
exits without writing the artifact. The cell at the bottom prints a
clear PASS / FAIL banner so it's unambiguous from a quick glance.

## Outputs (only on PASS)

- `/.../data/spatial_split/asset_id_to_split_v1.parquet`
- `/.../data/spatial_split/verification_world_scatter.png`

## Methodology specs (locked)

- Block size: 200 km × 200 km
- Projection: EPSG:8857 (Equal Earth)
- Block grid corner-anchored at EE (0, 0)
- Split target: 70 / 15 / 15
- Stratification: by region (not class) — block-level assignment
- Block assignment seed: 42 (invariant across FMs / training seeds)


In [13]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [14]:
%%capture
!pip install -q pyproj pyarrow pandas matplotlib geopandas scikit-learn


In [15]:
# Extract curation code zip to expose curation.utils.spatial_blocking.
# If the zip on Drive is older than this notebook (i.e. it doesn't yet
# include spatial_blocking.py), the import below will fail and the
# fallback cell appends the module inline.
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')
    else:
        print(f'WARNING: no zip at {CODE_ZIP}. Will fall back to inline module.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# Try the canonical import first. If the imported assign_blocks_to_splits
# is missing the class_fallback parameter (i.e. the zip predates the
# remediation), force the inline fallback path instead so the right
# behaviour is available regardless of which Drive zip we land on.
USE_INLINE_FALLBACK = False
try:
    from curation.utils.spatial_blocking import (
        compute_spatial_blocks,
        assign_blocks_to_splits,
        save_split_artifact,
        load_split_artifact,
    )
    import inspect
    if 'class_fallback' not in inspect.signature(assign_blocks_to_splits).parameters:
        print('Imported assign_blocks_to_splits lacks class_fallback param — '
              'falling back to inline definition.')
        USE_INLINE_FALLBACK = True
    else:
        print('Imported curation.utils.spatial_blocking with class_fallback support.')
except ImportError as e:
    print(f'Could not import from zip ({e}). Falling back to inline module.')
    USE_INLINE_FALLBACK = True

if USE_INLINE_FALLBACK:
    # ----------------------- Inline fallback (with class_fallback) -------
    # Mirrors curation/utils/spatial_blocking.py — keep them in sync when
    # the canonical file is edited.
    import random
    import numpy as np
    import pandas as pd
    from typing import Dict, List

    DEFAULT_PROJECTION    = 'EPSG:8857'
    DEFAULT_BLOCK_SIZE_KM = 200

    def compute_spatial_blocks(tiles_df, block_size_km=DEFAULT_BLOCK_SIZE_KM,
                               projection=DEFAULT_PROJECTION):
        from pyproj import Transformer
        if 'lat' not in tiles_df.columns or 'lon' not in tiles_df.columns:
            raise ValueError("tiles_df must have 'lat' and 'lon' columns")
        block_size_m = block_size_km * 1000
        transformer = Transformer.from_crs('EPSG:4326', projection, always_xy=True)
        lons = tiles_df['lon'].to_numpy(dtype=np.float64)
        lats = tiles_df['lat'].to_numpy(dtype=np.float64)
        ee_x, ee_y = transformer.transform(lons, lats)
        block_id_x = np.floor(ee_x / block_size_m).astype(np.int64)
        block_id_y = np.floor(ee_y / block_size_m).astype(np.int64)
        out = tiles_df.copy()
        out['ee_x']       = ee_x
        out['ee_y']       = ee_y
        out['block_id_x'] = block_id_x
        out['block_id_y'] = block_id_y
        out['block_id']   = [f'bx_{x}_by_{y}' for x, y in zip(block_id_x, block_id_y)]
        return out

    def assign_blocks_to_splits(tiles_with_blocks, train_frac=0.70,
                                val_frac=0.15, test_frac=0.15, seed=42,
                                stratify_by='region', class_fallback=None):
        if abs((train_frac + val_frac + test_frac) - 1.0) > 1e-6:
            raise ValueError('fractions must sum to 1.0')
        rng = random.Random(seed)
        block_split_lookup = {}
        for stratum_val, stratum_df in tiles_with_blocks.groupby(stratify_by, sort=True):
            block_sizes = (stratum_df.groupby('block_id').size()
                                  .sort_index().to_dict())
            block_ids = list(block_sizes.keys())
            rng.shuffle(block_ids)
            n_tiles_in_stratum = sum(block_sizes.values())
            targets = {'train': train_frac * n_tiles_in_stratum,
                       'val':   val_frac   * n_tiles_in_stratum,
                       'test':  test_frac  * n_tiles_in_stratum}
            running = {'train': 0, 'val': 0, 'test': 0}
            for bid in block_ids:
                order = ['train', 'val', 'test']
                best = max(order, key=lambda s: (targets[s] - running[s], -order.index(s)))
                block_split_lookup[bid] = best
                running[best] += block_sizes[bid]
        out = tiles_with_blocks.copy()
        out['split'] = out['block_id'].map(block_split_lookup)
        out['split_protocol'] = 'spatial_block'

        if class_fallback:
            for cls in class_fallback:
                mask = out['asset_type'] == cls
                n_cls = int(mask.sum())
                if n_cls == 0:
                    raise ValueError(
                        f'class_fallback class {cls!r} has 0 tiles in the dataset; '
                        f'check asset_type spelling'
                    )
                cls_ids = out.loc[mask, 'asset_id'].tolist()
                cls_rng = random.Random(seed)
                shuffled = cls_ids.copy()
                cls_rng.shuffle(shuffled)
                n_tr = int(n_cls * train_frac)
                n_va = int(n_cls * val_frac)
                fb_assignment = {}
                for i, aid in enumerate(shuffled):
                    if i < n_tr:
                        fb_assignment[aid] = 'train'
                    elif i < n_tr + n_va:
                        fb_assignment[aid] = 'val'
                    else:
                        fb_assignment[aid] = 'test'
                new_splits = out.loc[mask, 'asset_id'].map(fb_assignment)
                out.loc[mask, 'split']          = new_splits.values
                out.loc[mask, 'split_protocol'] = 'class_fallback'
        return out

    SPLIT_ARTIFACT_COLUMNS = (
        'asset_id', 'region', 'sector', 'asset_type',
        'lat', 'lon', 'block_id_x', 'block_id_y',
        'split', 'split_protocol',
    )

    def save_split_artifact(tiles_with_splits, output_path):
        out = tiles_with_splits[list(SPLIT_ARTIFACT_COLUMNS)].copy()
        out['asset_id']       = out['asset_id'].astype(str)
        out['split']          = out['split'].astype(str)
        out['split_protocol'] = out['split_protocol'].astype(str)
        out['block_id_x']     = out['block_id_x'].astype('int64')
        out['block_id_y']     = out['block_id_y'].astype('int64')
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        out.to_parquet(output_path, index=False)

    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))

    print('Inline fallback module installed (with class_fallback support).')


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip -> /content/infrabench_repo ...
done.
Could not import from zip (No module named 'curation.utils.spatial_blocking'). Falling back to inline module.
Inline fallback module installed (with class_fallback support).


In [16]:
# ---- Constants & paths -----------------------------------------------------
DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
ALPHAEARTH_PARQUET  = f'{DRIVE_ROOT}/data/alphaearth/embeddings_2024.parquet'
SPLIT_DATA_DIR      = f'{DRIVE_ROOT}/data/spatial_split'
SPLIT_ARTIFACT_PATH = f'{SPLIT_DATA_DIR}/asset_id_to_split_v1.parquet'
SCATTER_PNG_PATH    = f'{SPLIT_DATA_DIR}/verification_world_scatter.png'
os.makedirs(SPLIT_DATA_DIR, exist_ok=True)

# ---- Methodology specs (locked) -------------------------------------------
BLOCK_SIZE_KM     = 200
PROJECTION        = 'EPSG:8857'
TRAIN_FRAC        = 0.70
VAL_FRAC          = 0.15
TEST_FRAC         = 0.15
BLOCK_ASSIGN_SEED = 42

# ---- Per-class fallback (remediation for the n<=12 tail) ------------------
# These two classes have tile counts (12 and 11 respectively) that fit
# inside a single 200 km block geographically — block-level assignment
# can't guarantee coverage across train/val/test. For these classes only,
# fall back to a per-class random stratified 70/15/15 split with the same
# seed=42 used for block assignment. All other classes remain on the
# spatial block protocol.
#
# Trade-off: tiles in these two classes lose block-coherence (same-block
# tiles may land in different splits). We accept this for ~23 tiles in
# order to guarantee class coverage; reviewers should be aware of the
# distinction (see split_protocol column in the saved parquet).
CLASS_FALLBACK = [
    'energy.generation.wind_farm',
    'transport.port_terminal',
]

# ---- Ontology (locked, must match all FM notebooks) -----------------------
REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']
CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.treatment.plant',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':                  'water.treatment.plant',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}

# Verify class_fallback entries are real classes
for cls in CLASS_FALLBACK:
    assert cls in CLASS_NAMES, f'CLASS_FALLBACK entry not in CLASS_NAMES: {cls!r}'

print(f'block_size_km:    {BLOCK_SIZE_KM}')
print(f'projection:       {PROJECTION}')
print(f'split target:     {TRAIN_FRAC} / {VAL_FRAC} / {TEST_FRAC}')
print(f'seed (blocks):    {BLOCK_ASSIGN_SEED}')
print(f'class fallback:   {CLASS_FALLBACK}')
print(f'output:           {SPLIT_ARTIFACT_PATH}')


block_size_km:    200
projection:       EPSG:8857
split target:     0.7 / 0.15 / 0.15
seed (blocks):    42
class fallback:   ['energy.generation.wind_farm', 'transport.port_terminal']
output:           /content/drive/MyDrive/infra_fm/data/spatial_split/asset_id_to_split_v1.parquet


In [17]:
# Build the canonical asset table from the 28 v1_1k manifests on Drive.
# Includes lat/lon (centroid), region, sector, asset_type (mapped to v1 canon).

import json, re, zipfile
import pandas as pd
from pathlib import Path

MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')
drive_path = Path(DATASETS_DRIVE)
assert drive_path.exists(), f'{DATASETS_DRIVE} not found'


def manifest_records_on_drive():
    """Yield (region, sector, manifest_dict) for every v1_1k cell — folder or zip."""
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m:
            continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS:
            continue
        if entry.is_dir():
            mp = entry / 'manifest.json'
            if not mp.exists():
                continue
            with mp.open() as f:
                yield region, sector, json.load(f)
            continue
        if entry.suffix == '.zip':
            folder_name = f'dataset_{region}_{sector}_v1_1k'
            with zipfile.ZipFile(entry) as zf:
                for cand in (f'{folder_name}/manifest.json', 'manifest.json'):
                    try:
                        yield region, sector, json.loads(zf.read(cand).decode('utf-8'))
                        break
                    except KeyError:
                        continue


rows = []
n_dropped_label = 0
n_dropped_geom  = 0
for region, sector, manifest in manifest_records_on_drive():
    for r in manifest.get('records', []):
        at = r.get('asset_type')
        if at not in ASSET_TYPE_MAP:
            n_dropped_label += 1
            continue
        lat, lon = r.get('lat'), r.get('lon')
        if lat is None or lon is None:
            n_dropped_geom += 1
            continue
        rows.append({
            'asset_id':   str(r.get('asset_id', '')),
            'region':     region,
            'sector':     sector,
            'asset_type': ASSET_TYPE_MAP[at],
            'lat':        float(lat),
            'lon':        float(lon),
        })

assets_df = pd.DataFrame(rows).drop_duplicates(subset=['asset_id']).reset_index(drop=True)
print(f'Built asset table: {len(assets_df):,} tiles')
print(f'  dropped (asset_type not in ASSET_TYPE_MAP): {n_dropped_label}')
print(f'  dropped (missing lat/lon):                  {n_dropped_geom}')
print()
print('Per-region:')
print(assets_df.groupby('region').size().to_string())
print('\nPer-class:')
print(assets_df.groupby('asset_type').size().sort_values(ascending=False).to_string())


Built asset table: 18,750 tiles
  dropped (asset_type not in ASSET_TYPE_MAP): 0
  dropped (missing lat/lon):                  0

Per-region:
region
africa               2842
asia                 2765
australia-oceania    3014
central-america      2565
europe               1737
north-america        2823
south-america        3004

Per-class:
asset_type
transport.train_station           5046
water.storage_tank                3993
energy.distribution.other         3264
water.wastewater.plant            1299
energy.distribution.substation     975
water.treatment.plant              895
transport.airport                  871
energy.transmission.substation     676
energy.generation.solar_farm       631
energy.generation.power_plant      545
telecom.data_center                532
energy.generation.wind_farm         12
transport.port_terminal             11


In [18]:
# Drop tiles missing from the AlphaEarth embeddings parquet so all four
# FMs evaluate on exactly the same set. Report distribution of dropped
# tiles — if they're concentrated in a rare class, surface that.

import pandas as pd

ae_path = Path(ALPHAEARTH_PARQUET)
if not ae_path.exists():
    raise RuntimeError(
        f'AlphaEarth parquet missing: {ALPHAEARTH_PARQUET}. '
        'Run the AlphaEarth fetch notebook first.'
    )

ae_ids = set(pd.read_parquet(ae_path, columns=['asset_id'])['asset_id'].astype(str))
print(f'AlphaEarth parquet covers {len(ae_ids):,} asset_ids')

asset_ids_all = set(assets_df['asset_id'])
missing_in_ae = asset_ids_all - ae_ids
print(f'Tiles in master list NOT in AlphaEarth parquet: {len(missing_in_ae):,}')

if missing_in_ae:
    missing_df = assets_df[assets_df['asset_id'].isin(missing_in_ae)].copy()
    print('\nDropped-tile distribution:')
    print('  by region:')
    print(missing_df.groupby('region').size().to_string())
    print('  by sector:')
    print(missing_df.groupby('sector').size().to_string())
    print('  by class:')
    print(missing_df.groupby('asset_type').size().to_string())
    rare = ['telecom.data_center', 'energy.generation.wind_farm',
            'transport.port_terminal']
    for cls in rare:
        n_total_cls   = (assets_df['asset_type'] == cls).sum()
        n_dropped_cls = (missing_df['asset_type'] == cls).sum()
        if n_dropped_cls > 0:
            pct = 100.0 * n_dropped_cls / max(n_total_cls, 1)
            print(f'  RARE-CLASS HIT: {cls}  dropped {n_dropped_cls}/{n_total_cls} '
                  f'({pct:.1f}%)')

assets_df = assets_df[~assets_df['asset_id'].isin(missing_in_ae)].reset_index(drop=True)
print(f'\nFinal master tile set: {len(assets_df):,} tiles  '
      f'(was {len(assets_df) + len(missing_in_ae)})')


AlphaEarth parquet covers 18,750 asset_ids
Tiles in master list NOT in AlphaEarth parquet: 0

Final master tile set: 18,750 tiles  (was 18750)


In [19]:
# Compute Equal Earth blocks, then assign blocks -> splits (with
# per-class fallback override for the rare-class tail).
blocked = compute_spatial_blocks(
    assets_df, block_size_km=BLOCK_SIZE_KM, projection=PROJECTION
)
print(f'Computed spatial blocks: {blocked["block_id"].nunique():,} unique blocks')
print(f'  ee_x range: [{blocked["ee_x"].min():,.0f}, {blocked["ee_x"].max():,.0f}] m')
print(f'  ee_y range: [{blocked["ee_y"].min():,.0f}, {blocked["ee_y"].max():,.0f}] m')
print(f'  block_id_x range: [{blocked["block_id_x"].min()}, {blocked["block_id_x"].max()}]')
print(f'  block_id_y range: [{blocked["block_id_y"].min()}, {blocked["block_id_y"].max()}]')

print('\nBlocks per region:')
print(blocked.groupby('region')['block_id'].nunique().to_string())

block_sizes = blocked.groupby('block_id').size()
print(f'\nTiles-per-block distribution:')
print(f'  min:    {block_sizes.min()}')
print(f'  median: {int(block_sizes.median())}')
print(f'  max:    {block_sizes.max()}')
print(f'  mean:   {block_sizes.mean():.1f}')

splits_df = assign_blocks_to_splits(
    blocked,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    seed=BLOCK_ASSIGN_SEED, stratify_by='region',
    class_fallback=CLASS_FALLBACK,
)
print(f'\nAssigned splits: {dict(splits_df.groupby("split").size())}')
print(f'Protocols:       {dict(splits_df.groupby("split_protocol").size())}')


Computed spatial blocks: 2,209 unique blocks
  ee_x range: [-16,666,966, 16,828,489] m
  ee_y range: [-6,467,044, 7,717,897] m
  block_id_x range: [-84, 84]
  block_id_y range: [-33, 38]

Blocks per region:
region
africa               425
asia                 628
australia-oceania    233
central-america       66
europe               223
north-america        344
south-america        321

Tiles-per-block distribution:
  min:    1
  median: 3
  max:    494
  mean:   8.5

Assigned splits: {'test': np.int64(2812), 'train': np.int64(13087), 'val': np.int64(2851)}
Protocols:       {'class_fallback': np.int64(23), 'spatial_block': np.int64(18727)}


In [20]:
# Verification 1: per-region distribution.
import pandas as pd

VERIFICATION_PASSED = True       # flipped to False if any check fails
VERIFICATION_NOTES  = []

print('=' * 76)
print('VERIFICATION 1: Per-region split distribution')
print('=' * 76)
print(f'{"region":<22s} {"n":>6s} {"train":>9s} {"val":>9s} {"test":>9s}  '
      f'{"tr%":>5s} {"va%":>5s} {"te%":>5s}')
print('-' * 76)
per_region = splits_df.groupby(['region', 'split']).size().unstack(fill_value=0)
per_region = per_region.reindex(columns=['train', 'val', 'test'], fill_value=0)
worst_dev = 0.0
for region in REGIONS:
    n = per_region.loc[region].sum() if region in per_region.index else 0
    tr = per_region.loc[region, 'train'] if region in per_region.index else 0
    va = per_region.loc[region, 'val']   if region in per_region.index else 0
    te = per_region.loc[region, 'test']  if region in per_region.index else 0
    trp = (100.0 * tr / n) if n else 0
    vap = (100.0 * va / n) if n else 0
    tep = (100.0 * te / n) if n else 0
    print(f'{region:<22s} {n:>6d} {tr:>9d} {va:>9d} {te:>9d}  '
          f'{trp:>5.1f} {vap:>5.1f} {tep:>5.1f}')
    worst_dev = max(worst_dev, abs(trp - 70), abs(vap - 15), abs(tep - 15))
print('-' * 76)
total = per_region.values.sum()
gtr = per_region['train'].sum()
gva = per_region['val'].sum()
gte = per_region['test'].sum()
print(f'{"TOTAL":<22s} {total:>6d} {gtr:>9d} {gva:>9d} {gte:>9d}  '
      f'{100.0*gtr/total:>5.1f} {100.0*gva/total:>5.1f} {100.0*gte/total:>5.1f}')
print(f'\nWorst per-region deviation from target: {worst_dev:.1f} pp')
if worst_dev > 10:
    VERIFICATION_NOTES.append(
        f'per-region: worst within-region deviation from 70/15/15 = '
        f'{worst_dev:.1f}pp (>10pp threshold)'
    )
    # 10pp is a soft cap — the small regions can drift a few pp; don't
    # block the run for this. Just record it.
print(f'(soft check only — block-level granularity makes some drift expected)')


VERIFICATION 1: Per-region split distribution
region                      n     train       val      test    tr%   va%   te%
----------------------------------------------------------------------------
africa                   2842      1984       424       434   69.8  14.9  15.3
asia                     2765      1914       418       433   69.2  15.1  15.7
australia-oceania        3014      2108       451       455   69.9  15.0  15.1
central-america          2565      1817       391       357   70.8  15.2  13.9
europe                   1737      1193       242       302   68.7  13.9  17.4
north-america            2823      1987       434       402   70.4  15.4  14.2
south-america            3004      2084       491       429   69.4  16.3  14.3
----------------------------------------------------------------------------
TOTAL                   18750     13087      2851      2812   69.8  15.2  15.0

Worst per-region deviation from target: 2.4 pp
(soft check only — block-level granularit

In [21]:
# Verification 2: per-class distribution — CRITICAL gate.
# After class_fallback override, every class should have >=1 in every split.
# Catalog any remaining failures and do not stop early.
import pandas as pd

print('=' * 90)
print('VERIFICATION 2: Per-class split distribution (with split protocol)')
print('=' * 90)
print(f'{"class":<36s} {"protocol":<16s} {"n":>6s} {"train":>9s} {"val":>9s} {"test":>9s}')
print('-' * 90)

# Determine protocol per class — every tile in a given class shares one
# protocol value, so groupby gives a single-element set per class.
class_protocol = (splits_df.groupby('asset_type')['split_protocol']
                           .agg(lambda s: ','.join(sorted(set(s))))
                           .to_dict())

per_class = splits_df.groupby(['asset_type', 'split']).size().unstack(fill_value=0)
per_class = per_class.reindex(columns=['train', 'val', 'test'], fill_value=0)

missing_class_split = []
for cls in CLASS_NAMES:
    proto = class_protocol.get(cls, '(no_tiles)')
    if cls not in per_class.index:
        for sp in ('train', 'val', 'test'):
            missing_class_split.append((cls, sp, 0, proto))
        print(f'{cls:<36s} {proto:<16s} {0:>6d} {0:>9d} {0:>9d} {0:>9d}')
        continue
    n = per_class.loc[cls].sum()
    tr = int(per_class.loc[cls, 'train'])
    va = int(per_class.loc[cls, 'val'])
    te = int(per_class.loc[cls, 'test'])
    marker = '  !! ' if 0 in (tr, va, te) else '     '
    print(f'{cls:<36s} {proto:<16s} {n:>6d} {tr:>9d} {va:>9d} {te:>9d}{marker}')
    for sp, count in [('train', tr), ('val', va), ('test', te)]:
        if count == 0:
            missing_class_split.append((cls, sp, n, proto))

print('-' * 90)
print(f'Spatial-block tiles:   {(splits_df["split_protocol"] == "spatial_block").sum():>6d}')
print(f'Class-fallback tiles:  {(splits_df["split_protocol"] == "class_fallback").sum():>6d}')

if missing_class_split:
    VERIFICATION_PASSED = False
    print()
    print('!! ' + '=' * 86)
    print('!! VERIFICATION 2 FAILED — classes still missing from at least one split')
    print('!! (this is unexpected after class_fallback override unless a fallback class')
    print('!!  has n<4 globally — investigate)')
    print('!! ' + '=' * 86)
    for cls, sp, n_global, proto in missing_class_split:
        print(f'    {cls:<36s} {proto:<16s} {sp:<8s} n_global={n_global}')
    VERIFICATION_NOTES.append(
        f'per-class (post-fallback): {len(missing_class_split)} (class, split) cells empty'
    )
else:
    print('Every class has >=1 tile in train, val, AND test (post-fallback). PASS.')


VERIFICATION 2: Per-class split distribution (with split protocol)
class                                protocol              n     train       val      test
------------------------------------------------------------------------------------------
energy.transmission.substation       spatial_block       676       459       112       105     
energy.distribution.substation       spatial_block       975       671       163       141     
energy.distribution.other            spatial_block      3264      2314       439       511     
energy.generation.power_plant        spatial_block       545       366       109        70     
energy.generation.solar_farm         spatial_block       631       432       101        98     
energy.generation.wind_farm          class_fallback       12         8         1         3     
water.wastewater.plant               spatial_block      1299       837       234       228     
water.treatment.plant                spatial_block       895       659       11

In [22]:
# Verification 3: compare against the old random stratified split.
# Regenerate the OLD split in-notebook by replicating the v1 CROMA logic
# exactly (per-(region, sector) cell, stratified_split(seed=42) by class).
import random

def old_stratified_split(rows, train_frac=0.7, val_frac=0.15, seed=42):
    """Mirror of CROMA v1's stratified_split, applied to a list of
    (cell-local position, class label) tuples. Returns three lists of
    cell-local positions for train/val/test."""
    by_class = {}
    for i, label in rows:
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


# Walk the master asset list in (region, sector) order — that's the order
# CROMA v1 builds source_datasets in (sorted iter over Drive paths -> the
# same alphabetical (region, sector) tuples). Each cell's records are also
# iterated in manifest order, which we approximate by using the order of
# rows in assets_df (which itself came from sorted Drive iteration).
old_split = {}                     # asset_id -> 'train'/'val'/'test'
for (region, sector), cell_df in assets_df.groupby(['region', 'sector'], sort=True):
    cell_df = cell_df.reset_index(drop=True)
    rows = [(i, CLASS_TO_IDX[c]) for i, c in enumerate(cell_df['asset_type'])]
    tr, va, te = old_stratified_split(rows)
    for i, sp in [(j, 'train') for j in tr] + [(j, 'val') for j in va] + [(j, 'test') for j in te]:
        old_split[cell_df.iloc[i]['asset_id']] = sp

# Now compare against the new spatial split.
splits_df['split_new'] = splits_df['split']
splits_df['split_old'] = splits_df['asset_id'].map(old_split)

# A handful of tiles may be unmapped if assets_df had duplicates that got
# squeezed somewhere. Sanity-check.
unmapped = splits_df['split_old'].isna().sum()
if unmapped > 0:
    print(f'WARNING: {unmapped} tiles could not be assigned an old-split — '
          f'comparison will exclude them.')

cmp_df = splits_df.dropna(subset=['split_old'])
n = len(cmp_df)
print('=' * 76)
print(f'VERIFICATION 3: Old (random) vs new (spatial) split comparison '
      f'({n:,} tiles)')
print('=' * 76)
transition_counts = (cmp_df.groupby(['split_old', 'split_new'])
                            .size()
                            .unstack(fill_value=0))
transition_counts = transition_counts.reindex(
    index=['train', 'val', 'test'], columns=['train', 'val', 'test'],
    fill_value=0,
)
print('\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ['train', 'val', 'test']:
    row = transition_counts.loc[old_sp]
    print(f'{old_sp:<8s}  {row["train"]:>10d} {row["val"]:>9d} {row["test"]:>10d}')

same = sum(transition_counts.loc[s, s] for s in ['train', 'val', 'test'])
changed = n - same
print(f'\nUnchanged: {same:,} ({100.0*same/n:.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/n:.1f}%)')


VERIFICATION 3: Old (random) vs new (spatial) split comparison (18,750 tiles)

old \ new      train       val       test
--------------------------------------------------
train           9125      1995       1960
val             1948       409        416
test            2014       447        436

Unchanged: 9,970 (53.2%)
Changed:   8,780 (46.8%)


In [23]:
# Verification 4: world scatter coloured by split (+ optional country outlines).
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(16, 8))

# Optional country outlines underneath (geopandas natural-earth_lowres ships
# with geopandas <1.0; in newer geopandas this is removed).
try:
    import geopandas as gpd
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    world.boundary.plot(ax=ax, color='lightgray', linewidth=0.5, zorder=1)
    print('  country outlines: drawn via geopandas naturalearth_lowres')
except Exception as e:
    print(f'  country outlines: skipped ({e.__class__.__name__})')

COLORS = {'train': 'tab:blue', 'val': 'tab:orange', 'test': 'tab:red'}
for sp in ['train', 'val', 'test']:
    sub = splits_df[splits_df['split'] == sp]
    ax.scatter(sub['lon'], sub['lat'],
               s=2, c=COLORS[sp], alpha=0.55, label=f'{sp} (n={len(sub):,})',
               zorder=2)

ax.set_xlim(-180, 180); ax.set_ylim(-60, 75)
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title(f'Infra-Bench v1 spatial split — {len(splits_df):,} tiles, '
             f'{BLOCK_SIZE_KM} km blocks, EPSG:8857, seed={BLOCK_ASSIGN_SEED}')
ax.legend(loc='lower left', framealpha=0.9, markerscale=3)
ax.grid(True, alpha=0.25, linewidth=0.5)
plt.tight_layout()
plt.savefig(SCATTER_PNG_PATH, dpi=140, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {SCATTER_PNG_PATH}')


  country outlines: skipped (AttributeError)
Saved: /content/drive/MyDrive/infra_fm/data/spatial_split/verification_world_scatter.png


In [24]:
# Final gate: write the split parquet only if all verifications passed.
print('=' * 76)
print(f'OVERALL VERIFICATION: {"PASS" if VERIFICATION_PASSED else "FAIL"}')
print('=' * 76)
if VERIFICATION_NOTES:
    print('Notes:')
    for note in VERIFICATION_NOTES:
        print(f'  - {note}')
print()

# Protocol summary
proto_counts = splits_df['split_protocol'].value_counts().to_dict()
total = int(splits_df.shape[0])
print('Split-protocol summary:')
print(f'  spatial_block:   {proto_counts.get("spatial_block", 0):>6d}  '
      f'({100.0 * proto_counts.get("spatial_block", 0) / total:.2f}%)')
print(f'  class_fallback:  {proto_counts.get("class_fallback", 0):>6d}  '
      f'({100.0 * proto_counts.get("class_fallback", 0) / total:.2f}%)')
print(f'  TOTAL:           {total:>6d}')
print()
print('Tiles per (protocol, split):')
for proto in ('spatial_block', 'class_fallback'):
    row = splits_df[splits_df['split_protocol'] == proto].groupby('split').size()
    row = row.reindex(['train', 'val', 'test'], fill_value=0)
    print(f'  {proto:<16s} train={int(row["train"]):>6d}  '
          f'val={int(row["val"]):>5d}  test={int(row["test"]):>5d}')
print()

if VERIFICATION_PASSED:
    save_split_artifact(splits_df, SPLIT_ARTIFACT_PATH)
    print(f'Split artifact written: {SPLIT_ARTIFACT_PATH}')
    print(f'  columns: asset_id, region, sector, asset_type, lat, lon, '
          f'block_id_x, block_id_y, split, split_protocol')
    print(f'  rows:    {len(splits_df):,}')
    print()
    print('Phase 1 verification complete. Phase 2 FM notebooks can now load '
          'this artifact via load_split_artifact(SPLIT_ARTIFACT_PATH).')
else:
    print('Split artifact NOT written. Address the failures above before '
          're-running this notebook.')


OVERALL VERIFICATION: PASS

Split-protocol summary:
  spatial_block:    18727  (99.88%)
  class_fallback:      23  (0.12%)
  TOTAL:            18750

Tiles per (protocol, split):
  spatial_block    train= 13072  val= 2849  test= 2806
  class_fallback   train=    15  val=    2  test=    6

Split artifact written: /content/drive/MyDrive/infra_fm/data/spatial_split/asset_id_to_split_v1.parquet
  columns: asset_id, region, sector, asset_type, lat, lon, block_id_x, block_id_y, split, split_protocol
  rows:    18,750

Phase 1 verification complete. Phase 2 FM notebooks can now load this artifact via load_split_artifact(SPLIT_ARTIFACT_PATH).
